In [31]:


!curl -L -o models/yolo26m-seg-hand.pt https://huggingface.co/NightingaleCen/YOLO26m-seg-hand/resolve/main/yolo26m-seg-hand.pt

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100   1052 100   1052   0      0   3600      0                              0
100 51.90M 100 51.90M   0      0  7.30M      0   00:07   00:07          7.73M


In [32]:
from pathlib import Path

model_path = Path("models/yolo26m-seg-hand.pt")

print("Datei existiert:", model_path.exists())

if model_path.exists():
    print("Dateigröße:", model_path.stat().st_size, "Bytes")

    with open(model_path, "rb") as f:
        print("Erste Bytes:", f.read(100))

Datei existiert: True
Dateigröße: 54421937 Bytes
Erste Bytes: b'PK\x03\x04\x00\x00\x08\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\r\x00\x15\x00best/data.pklFB\x11\x00ZZZZZZZZZZZZZZZZZ\x80\x02}q\x00(X\x04\x00\x00\x00dateq\x01X\x1a\x00\x00\x002026-07-03T17:'


In [33]:
from ultralytics import YOLO
model = YOLO('models/yolo26m-seg-hand.pt')

In [77]:
import cv2
from PIL import Image
from numpy import asarray

In [111]:
image_path = "raw_data/MINST_pre_pro/Dataset/user_3/A4.jpg"
image = Image.open(image_path)
results = model.predict(image_path, save=True)


image 1/1 /home/maxgr/code/notFonz/HandSpell/raw_data/MINST_pre_pro/Dataset/user_3/A4.jpg: 192x224 1 hand, 5189.1ms
Speed: 76.6ms preprocess, 5189.1ms inference, 73.9ms postprocess per image at shape (1, 3, 192, 224)
Results saved to /home/maxgr/code/notFonz/HandSpell/runs/segment/predict


In [112]:
for object in results:
    print(object.boxes.xyxy)

tensor([[132.4831,  33.7271, 208.6278,  98.5632]])


In [113]:
x1, y1, x2, y2 = object.boxes.xyxy[0]

In [114]:
height = y2 -y1
width = x2 - x1

In [ ]:
res = image.crop((int(x1), int(y1), int(x2), int(y2)))


In [117]:
height = y2 -y1
width = x2 - x1
target_size = 64
#Determine scaling factor
scale = target_size/ max(height, width)

In [118]:
new_width = width*scale
new_height = height*scale

In [120]:
numpy_image = asarray(res)

In [121]:
resized_image = cv2.resize(numpy_image, (int(new_width), int(new_height)), interpolation=cv2.INTER_AREA)

In [122]:
pad_width = target_size - new_width
pad_height = target_size - new_height

In [123]:
top = pad_height // 2
bottom = pad_height -top
left = pad_width // 2
right = pad_width -left

In [126]:
normalized_image = cv2.copyMakeBorder(resized_image, int(top), int(bottom), int(left), int(right), cv2.BORDER_CONSTANT, value=(255,255,255))

In [127]:
normalized_image1 = Image.fromarray(normalized_image)

In [128]:
normalized_image1.show()